# 07 - Monitoring Refresh (Healthcare)
Simple monitoring notebook for freshness, rejected records and pipeline run history.

## 1. Setup

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "capstone_8hospital")
dbutils.widgets.text("run_id", "manual")

CATALOG = dbutils.widgets.get("catalog")
RUN_ID = dbutils.widgets.get("run_id")

BRONZE = f"{CATALOG}.bronze_schema"
SILVER = f"{CATALOG}.silver_schema"
GOLD = f"{CATALOG}.gold_schema"
OPS = f"{CATALOG}.ops_schema"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OPS}")

def count(table):
    return spark.table(table).count()

def hours_since_write(table):
    last = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["timestamp"]
    now = spark.sql("SELECT current_timestamp()").first()[0]
    return round((now - last).total_seconds() / 3600, 2)

print("Catalog:", CATALOG)
print("Run ID:", RUN_ID)

## 2. Table Freshness
Check row count, last write time, last operation and hours since the last write.

In [0]:
tables = {
    "bronze": ["bronze_patients", "bronze_departments", "bronze_beds", "bronze_patient_encounters", "bronze_lab_test_events", "bronze_bed_transfer_events"],
    "silver": ["silver_patients", "silver_departments", "silver_beds", "silver_patient_encounters", "silver_lab_test_events", "silver_bed_transfer_events", "silver_rejected_records", "silver_duplicate_audit"],
    "gold": ["dim_patient", "dim_department", "dim_bed", "dim_date", "fact_patient_encounter", "fact_lab_test_event", "fact_bed_transfer_event", "mart_department_daily"]
}

rows = []
for layer, names in tables.items():
    for name in names:
        table = f"{BRONZE if layer == 'bronze' else SILVER if layer == 'silver' else GOLD}.{name}"
        last = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()
        rows.append((layer, name, count(table), last["timestamp"], last["operation"], hours_since_write(table)))

freshness = spark.createDataFrame(
    rows,
    "layer string, table_name string, row_count long, last_write_ts timestamp, last_operation string, hours_since_write double"
).withColumn("refreshed_ts", F.current_timestamp())

freshness.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.monitor_table_freshness")

print("Table freshness refreshed")
display(freshness.orderBy("layer", "table_name"))

## 3. Rejected Records
Show rejected rows by dataset and validation rule, with reject percentage against Bronze rows.

In [0]:
source_tables = {
    "patient_encounters": "bronze_patient_encounters",
    "lab_test_events": "bronze_lab_test_events",
    "bed_transfer_events": "bronze_bed_transfer_events",
    "patient_master": "bronze_patients",
    "department_master": "bronze_departments",
    "bed_master": "bronze_beds"
}

bronze_rows = spark.createDataFrame(
    [(dataset, table, count(f"{BRONZE}.{table}") ) for dataset, table in source_tables.items()],
    "dataset string, source_table string, bronze_rows long"
)

rejected = (
    spark.table(f"{SILVER}.silver_rejected_records")
    .groupBy("dataset", "rule_id", "reason")
    .agg(F.count("*").alias("rejected_rows"))
    .join(bronze_rows, "dataset", "left")
    .withColumn("reject_pct", F.round(100 * F.col("rejected_rows") / F.col("bronze_rows"), 2))
    .withColumn("refreshed_ts", F.current_timestamp())
)

rejected.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.monitor_rejected_records")

print("Rejected-record monitoring refreshed")
display(rejected.orderBy("dataset", "rule_id"))

## 4. Pipeline Run History
Store one monitoring row for each pipeline run.

In [0]:
checks = spark.table(f"{OPS}.pipeline_validation_results").filter(F.col("run_id") == RUN_ID)
status_counts = {r["status"]: r["n"] for r in checks.groupBy("status").agg(F.count("*").alias("n")).collect()}

passed = status_counts.get("PASS", 0)
warned = status_counts.get("WARN", 0)
failed = status_counts.get("FAIL", 0)

if failed:
    overall = "FAILED"
else:
    overall = "SUCCESS"

run_row = spark.createDataFrame([(
    RUN_ID,
    spark.sql("SELECT current_timestamp()").first()[0],
    passed,
    warned,
    failed,
    overall,
    count(f"{SILVER}.silver_rejected_records"),
    count(f"{SILVER}.silver_duplicate_audit"),
    count(f"{GOLD}.fact_patient_encounter"),
    count(f"{GOLD}.mart_department_daily")
)], "run_id string, run_ts timestamp, checks_passed long, checks_warned long, checks_failed long, overall_status string, rejected_records long, duplicate_records long, fact_encounter_rows long, mart_rows long")

run_row.write.mode("append").saveAsTable(f"{OPS}.monitor_pipeline_runs")

print("Pipeline run recorded")
print("Overall status:", overall)
display(spark.table(f"{OPS}.monitor_pipeline_runs").orderBy(F.desc("run_ts")))

## 5. Final Check

In [0]:
print("Monitoring refresh completed")
print("Freshness table:", f"{OPS}.monitor_table_freshness")
print("Rejected-record table:", f"{OPS}.monitor_rejected_records")
print("Run-history table:", f"{OPS}.monitor_pipeline_runs")
print("Status:", overall)